# hpDIRC starter notebook

This notebook reads data from the hpDIRC test stand and does a first, simple analysis.
Run the cells from top to bottom with **Shift+Enter**.

**What's in the data.** The detector signals are digitised by a CAEN V1742 board: **32 channels**
in **4 groups of 8**, each recording **1024 samples** at 5 GS/s (one sample every 0.2 ns,
so ~205 ns per event). Each group also records the fast trigger signal (TR0/TR1).
At the start of each run the DAQ also saves the setup: HV settings, motor position, etc.

**What the `hpdirc` module does for you.** It opens the raw `.evt` file and applies the
digitiser's calibration (`calib_0049_5G.dat`), so the waveforms you get are clean numpy arrays:

| | shape | units |
|---|---|---|
| `ev.wave` | (32, 1024) | ADC counts |
| `ev.time` | (32, 1024) | ns |
| `ev.trig` | (4, 1024) | ADC counts, trigger signal per group |

The signals are **negative-going** pulses below a baseline of roughly 2000 ADC counts.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hpdirc

# --- change these to the run you want to look at ---
DATA_FILE  = "data/test-00000068-0000.evt"
CALIB_FILE = "data/calib_0049_5G.dat"

## 1. Open a run and look at the run conditions

In [ ]:
run = hpdirc.Run(DATA_FILE, calib=CALIB_FILE)

print("Begin-run information stored in this file:", run.beginrun_packets())
print()
print(run.hv())
print(run.motor())

## 2. Look at a single event

`run.events()` gives you the events one at a time. `next(...)` takes just the first one.

In [ ]:
events = run.events()
ev = next(events)

print("run", ev.run, " event", ev.number, " samples", ev.samples)

fig, axes = plt.subplots(4, 8, figsize=(18, 8), sharex=True, sharey=True)
for ch, ax in enumerate(axes.flat):
    ax.plot(ev.time[ch], ev.wave[ch], lw=0.7)
    ax.set_title(f"ch {ch}", fontsize=9)
fig.supxlabel("time [ns]")
fig.supylabel("ADC counts")
fig.suptitle(f"run {ev.run}, event {ev.number}")
plt.tight_layout()
plt.show()

Most channels will be flat in most events - a signal only shows up in channels where a photon
was detected. Let's step through events until we find one with a clear pulse in a chosen channel,
and zoom in on it together with the trigger signal of the same group.

In [ ]:
CHANNEL = 20         # try other channels!
THRESHOLD = 30       # ADC counts below baseline

for ev in events:
    w = ev.wave[CHANNEL]
    baseline = np.median(w)
    if baseline - w.min() > THRESHOLD:
        break

group = CHANNEL // 8
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax1.plot(ev.time[CHANNEL], ev.wave[CHANNEL])
ax1.axhline(baseline, color="gray", ls="--", label="baseline")
ax1.axhline(baseline - THRESHOLD, color="red", ls=":", label="threshold")
ax1.set_ylabel("ADC counts")
ax1.set_title(f"event {ev.number}, channel {CHANNEL}")
ax1.legend()
ax2.plot(ev.trigtime[group], ev.trig[group], color="C1")
ax2.set_ylabel("trigger [ADC]")
ax2.set_xlabel("time [ns]")
plt.show()

## 3. Loop over many events

For every event and every channel we measure three simple things and store them in a table
(a pandas DataFrame - like a spreadsheet):

* **baseline** - median of the waveform
* **amplitude** - how far the lowest point goes below the baseline
* **peak time** - when the lowest point happens (ns)

We also measure the **trigger time** of each group (when the trigger signal crosses
half-way between its baseline and its extreme), so that pulse times can be measured
*relative to the trigger*, which removes event-to-event jitter.

In [ ]:
def trigger_time(t, w):
    # time where the trigger signal first crosses 50% of its swing
    base = np.median(w[:50])
    swing = w - base
    i_ext = np.argmax(np.abs(swing))
    half = swing[i_ext] / 2
    crossed = np.nonzero(np.sign(half) * swing[: i_ext + 1] >= abs(half))[0]
    return t[crossed[0]] if len(crossed) else np.nan

def analyse(ev):
    rows = []
    trig_t = [trigger_time(ev.trigtime[g], ev.trig[g]) for g in range(4)]
    baseline = np.median(ev.wave, axis=1)
    i_min = np.argmin(ev.wave, axis=1)
    for ch in range(32):
        rows.append({
            "event": ev.number,
            "channel": ch,
            "baseline": baseline[ch],
            "amplitude": baseline[ch] - ev.wave[ch, i_min[ch]],
            "peak_time": ev.time[ch, i_min[ch]],
            "trigger_time": trig_t[ch // 8],
        })
    return rows

N_EVENTS = 5000      # set to None to read the whole file (can take minutes for big files)

run = hpdirc.Run(DATA_FILE, calib=CALIB_FILE)    # re-open to start from the beginning
rows = []
for ev in run.events(max_events=N_EVENTS):
    rows.extend(analyse(ev))

df = pd.DataFrame(rows)
df["dt"] = df["peak_time"] - df["trigger_time"]
df

## 4. Which channels see signals?

In [ ]:
THRESHOLD = 30
hits = df[df.amplitude > THRESHOLD]
rate = hits.groupby("channel").size().reindex(range(32), fill_value=0) / df.event.nunique()

plt.figure(figsize=(10, 4))
plt.bar(rate.index, rate.values)
plt.xlabel("channel")
plt.ylabel(f"fraction of events with amplitude > {THRESHOLD}")
plt.xticks(range(32))
plt.show()

# the same thing as a 4 x 8 map (one row per digitiser group)
plt.figure(figsize=(8, 4))
plt.imshow(rate.values.reshape(4, 8), cmap="viridis")
plt.colorbar(label="hit fraction")
for ch in range(32):
    plt.text(ch % 8, ch // 8, ch, ha="center", va="center", color="w", fontsize=8)
plt.xlabel("channel within group"); plt.ylabel("group")
plt.show()

## 5. Amplitude and timing for one channel

In [ ]:
CHANNEL = 20
d = df[df.channel == CHANNEL]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.hist(d.amplitude, bins=150, range=(0, 1.1 * d.amplitude.max()))
ax1.axvline(THRESHOLD, color="red", ls=":")
ax1.set_yscale("log")
ax1.set_xlabel("amplitude [ADC counts]"); ax1.set_title(f"channel {CHANNEL}")

sig = d[d.amplitude > THRESHOLD]
centre = sig.dt.median()
ax2.hist(sig.dt, bins=100, range=(centre - 3, centre + 3))
ax2.set_xlabel("peak time - trigger time [ns]"); ax2.set_title(f"channel {CHANNEL}, amplitude > {THRESHOLD}")
plt.show()

core = sig.dt[abs(sig.dt - centre) < 3]
print(f"{len(sig)} pulses; {len(core)} within 3 ns of the peak: "
      f"mean {core.mean():.2f} ns, std {core.std():.3f} ns")

## 6. Save your results

Saving the table means you don't have to re-read the raw data next time.

In [ ]:
df.to_csv("results.csv", index=False)
# later:  df = pd.read_csv("results.csv")

## Things to try

1. Change `CHANNEL` and `THRESHOLD`. Where is the noise, where are the real pulses?
   Make a histogram of the **baseline** - is it the same in every channel?
2. The peak time is only as good as one sample (0.2 ns). Improve it: fit the pulse, or use
   **constant-fraction timing** (the time the pulse crosses e.g. 50% of its amplitude,
   interpolating between samples - see `trigger_time()` for a start).
3. Instead of the minimum, compute the **pulse charge**: the area below the baseline in a
   window around the peak. Plot the charge spectrum - can you see single photo-electrons?
4. Compare runs taken at different HV (look at `run.hv()`) or motor positions (`run.motor()`).
   How do the hit rate and amplitude change?
5. Look at events with signals in **several** channels at once. How are their times correlated?

**Help:** `help(hpdirc.Run)`. To see the waveforms without the DRS4 spike removal,
open the run with `hpdirc.Run(DATA_FILE, calib=CALIB_FILE, despike=False)` and look at
channels with no signal - you'll see the narrow spikes this step removes.